<h1 style=\"text-align: center; font-size: 50px;\"> Pre-trained BERT for building a Q&A system MLflow integration </h1>

Notebook Overview
- Start Execution
- User Constants
- Install and Import Libraries
- Configure Settings
- Logging Model to MLflow
- Fetching the Latest Model Version from MLflow
- Loading the Model and Running Inference

## Start Execution

In [1]:
%%time

%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.
CPU times: user 16.1 ms, sys: 15.6 ms, total: 31.6 ms
Wall time: 1.09 s


In [2]:
MIN_TOTAL_RAM_GB = 16
MIN_TOTAL_VRAM_GB = 4


from ai_studio_blueprint_kit.memory_guard import run_memory_check_notebook


run_memory_check_notebook(
    min_total_ram_gb=MIN_TOTAL_RAM_GB,
    min_total_vram_gb=MIN_TOTAL_VRAM_GB,
)

In [1]:
import logging
import time

# Configure logger
logger: logging.Logger = logging.getLogger("register_model_logger")
logger.setLevel(logging.INFO)
logger.propagate = False  # Prevent duplicate logs from parent loggers

# Set formatter
formatter: logging.Formatter = logging.Formatter(
    fmt="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

# Configure and attach stream handler
stream_handler: logging.StreamHandler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

In [2]:
start_time = time.time()  

logger.info("Notebook execution started.")

2026-04-15 02:32:27 - INFO - Notebook execution started.


## User Constants

In [3]:
CONTEXT = "Marta is mother of John and Amanda"
QUESTION = "what is the name of Marta's daugther?"

## Install and Import Libraries

In [5]:
# ------------------------ Standard Library Imports ------------------------
import warnings
from pathlib import Path
import shutil
import sys
import os

# ------------------------ MLflow for Experiment Tracking and Model Management ------------------------
import mlflow
from mlflow.types.schema import Schema, ColSpec
from mlflow.types import ParamSchema, ParamSpec
from mlflow.models import ModelSignature
from mlflow import MlflowClient

# ------------------------ Third-Party Libraries ------------------------
from transformers import pipeline
import torch

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from src.mlflow import Logger

from src.utils import (
    load_config,
)

from IPython import get_ipython

2026-04-15 02:32:32.002290: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-15 02:32:32.013743: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776220352.027656    2124 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776220352.032637    2124 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-15 02:32:32.049467: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

## Configure Settings

In [6]:
warnings.filterwarnings("ignore")

In [7]:
# ------------------------- Define global experiment and run names to be used throughout the notebook -------------------------

MODEL_CHECKPOINT = "distilbert-base-cased"
EXPERIMENT_SET = "BERT Q&A - distilbert-base-cased"
SAVE_MODEL_NAME = "distilbert_bertqa" if os.path.exists("distilbert_bertqa") else MODEL_CHECKPOINT
EXPERIMENT_NAME = "BERT model for Q&A"
MODEL_NAME = "BERT_QA"
RUN_NAME = 'BERT_QA'
NAME = 'BERT_QA'

# ------------------------- Paths -------------------------
DEMO_FOLDER = "../demo"
CONFIG_PATH = "../configs/config.yaml"

# ------------------------- Set up the chunk separator for text processing -------------------------
CHUNK_SEPARATOR = "\n\n"

## Logging Model to MLflow

In [8]:
# Create the question-answering pipeline for model training/saving
model_name = SAVE_MODEL_NAME
qa_pipeline = pipeline(
    'question-answering',
    model=model_name,
    device=0 if torch.cuda.is_available() else -1  # GPU if available, otherwise CPU
)

Device set to use cuda:0


In [9]:
# Define input/output signature for MLflow
input_schema = Schema([
    ColSpec("string", "context"),
    ColSpec("string", "question"),
])
output_schema = Schema([
    ColSpec("string", "answer")
])
params_schema = ParamSchema([
    ParamSpec("show_score", "boolean", False)
])
signature = ModelSignature(inputs=input_schema, outputs=output_schema, params=params_schema)

# Log model using the new models-from-code approach
mlflow.set_tracking_uri('/phoenix/mlflow')
mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

with mlflow.start_run(run_name=RUN_NAME) as run:
    logger.info(f"Run's Artifact URI: {run.info.artifact_uri}")
    
    # Use the new Logger to log the model
    Logger.log_model(
        signature=signature,
        artifact_path=MODEL_NAME,
        config_path=CONFIG_PATH,
        model_checkpoint=model_name,
        source_pipeline=qa_pipeline,
        demo_folder=DEMO_FOLDER
    )
    
    # Register the model
    mlflow.register_model(model_uri=f"runs:/{run.info.run_id}/{MODEL_NAME}", name=NAME)

2026/04/15 02:32:37 INFO mlflow.tracking.fluent: Experiment with name 'BERT model for Q&A' does not exist. Creating a new experiment.
2026-04-15 02:32:38 - INFO - Run's Artifact URI: /phoenix/mlflow/708210979097324831/5d0c830674934ab2b730226930cf12d1/artifacts
Successfully registered model 'BERT_QA'.
2026/04/15 02:32:49 WARNING mlflow.tracking._model_registry.fluent: Run with id 5d0c830674934ab2b730226930cf12d1 has no artifacts at artifact path 'BERT_QA', registering model based on models:/m-9ab3200913a64e73a9967ec3d681b304 instead
Created version '1' of model 'BERT_QA'.


## Fetching the Latest Model Version from MLflow

In [10]:
client = mlflow.MlflowClient()
model_metadata = client.get_latest_versions(MODEL_NAME, stages=["None"])
latest_model_version = model_metadata[0].version
print(latest_model_version, mlflow.models.get_model_info(f"models:/BERT_QA/{latest_model_version}").signature)

1 inputs: 
  ['context': string (required), 'question': string (required)]
outputs: 
  ['answer': string (required)]
params: 
  ['show_score': boolean (default: False)]



## Loading the Model and Running Inference

In [11]:
model = mlflow.pyfunc.load_model(model_uri=f"models:/BERT_QA/{latest_model_version}")
context = CONTEXT
question = QUESTION
model.predict({"context": [context], "question":[question]})

Device set to use cuda:0


{'answer': 'John and Amanda',
 'score': 0.37300360202789307,
 'start': 19,
 'end': 34}

In [12]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")

2026-04-15 02:32:55 - INFO - ⏱️ Total execution time: 0m 27.72s


In [1]:
status = "Notebook execution completed successfully"
print(f"Message: {status}")

Message: Notebook execution completed successfully


In [ ]:
app = get_ipython()
app.kernel.do_shutdown(restart=False)

Built with ❤️ using Z by HP AI Studio.